# 11.10 視覺 token 預算怎麼影響能力？

# 第 11 章：對齊、問答與語言能力保留

前置：第10章介面與第7章SFT。能力實驗需先有「文字屬性→正確答案」基模，以及可分類的視覺encoder；用scripts/prepare_data.py和pretrain_encoders.py準備。下面隨機模型只驗證資料／梯度，所有品質留給讀者訓練後量測。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：圖片切小方塊：先檢查能不能拼回來**

16×16 RGB，patch邊長4，得到16個patch。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/patchify.svg")))

**先想一想：**patch邊長減半，token數是兩倍嗎？

更小patch保留較細粒度的空間資訊，但增加視覺token。序列變長也增加自注意力工作；換patch size會改變encoder權重shape。

**把直覺寫成數學：**N=(H/P)(W/P)；完整attention矩陣元素約(T_text+N)²。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
h = 16
text_tokens = 20
for p in (8, 4, 2):
    n = (h // p) ** 2
    print(p, "視覺tokens", n, "總attention格數", (text_tokens + n) ** 2)

**如何讀結果：**換patch size不能直接加載舊patch投影參數；需對應checkpoint重新訓練比較。

**只改一件事：**只把圖像分辨率加倍，保持patch大小。
